In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from lib.data.dataloading import load_nursing_5_class
import torch
from torch import nn
from lib.config import *
import matplotlib.pyplot as plt
from lib.models import ResNetClassifierFiveClass, ResNetClassifierFiveClassXBlock, RegNet, RegNetNoX
from lib.modules import optimization_loop_multi_class

In [3]:
CONFIG = {
    'WINDOW_SIZE':3901,
    'WINDOW_STRIDE':3901,
    'BATCH_SIZE':128,
    'LEARNING_RATE':3e-4,
    'TEST_SIZE':0.2,
    'DEVICE':'cuda:1',
    'DEPTHI': [1],
    'WIDTHI': [64],
    'NTL': 2,
    'DMODEL': 256,
    'MASKPCT': 0.25
}

In [8]:
nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE'], 
    test_size=CONFIG['TEST_SIZE'], 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['WINDOW_STRIDE'],
)

[41, 52, 44, 54, 60, 18, 25, 43, 61, 40, 53, 65, 29, 67, 38, 26, 16, 42, 27, 62, 31, 63, 19, 24, 36, 48, 28, 56, 59, 68, 49, 12, 23, 57, 35, 17, 34, 47, 32, 30, 20, 50, 66, 14, 11, 64, 58, 55] [37, 46, 70, 39, 22, 13, 45, 69, 51, 33, 15, 21]


NameError: name 'DATA_DIR' is not defined

In [35]:
# model = ResNetClassifierFiveClass(WINSIZE, 3, (4,4,48)).to(DEVICE)
# model = RegNetNoX(WINSIZE, 3, 64, (1,), (64,)).to(DEVICE)
model = RegNet(WINSIZE, 3, 64, (2,), (64,), b=4, g=4).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()
print(sum([p.numel() for p in model.parameters() if p.requires_grad]))

32951


In [36]:
optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=1000,
    device=DEVICE,
    patience=200,
    # outdir='dev/test',
    # writer=f'runs/test-regnet/{str(type(model)).split(".")[-1][:-2]}_w{WINSIZE}_{model.dims_str}' + ('_p{model.p_dropout}' if model.p_dropout else '')
    # writer=f'runs/test-regnet/{str(type(model)).split(".")[-1][:-2]}_win{WINSIZE}_d{model.d_str}_w{model.w_str}' + ('_p{model.p_dropout}' if model.p_dropout else '')'
    writer=f'runs/test-regnet/{str(type(model)).split(".")[-1][:-2]}_win{WINSIZE}_d{model.d_str}_w{model.w_str}_so{model.stem_out_c}' + f'_b{model.b}_g{model.g}' if isinstance(model, RegNet) else '' + ('_p{model.p_dropout}' if model.p_dropout else '')
)

  0%|          | 0/1000 [00:00<?, ?it/s]

: Epoch 300: Train Loss: 0.14793: Dev Loss: 1.3286:  30%|███       | 300/1000 [04:59<11:39,  1.00it/s] 

Early stopping at epoch 300
